# Week 7 Exercise Notebook: AG News Topic Classification

In this notebook you will fine-tune a **multiclass news topic classifier** using the **AG News** dataset.

Our central question is:

**Does full fine-tuning improve enough over a frozen-head baseline to justify the extra training cost?**

By the end of this notebook, you should be able to:

- load and inspect the AG News dataset,
- prepare tokenized inputs for multiclass text classification,
- build a frozen-head baseline with `bert-base-uncased`,
- fully fine-tune the same checkpoint,
- compare both approaches using validation metrics,
- evaluate one chosen model on the test set.

## Setup

In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

import os
from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.getenv("HUGGINGFACE_API_KEY")

if not HF_TOKEN:
    raise ValueError(
       "HUGGINGFACE_API_KEY not found.\n"
        "→ Copy .env.example to .env\n"
        "→ Add your HuggingFace API key"
    )

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)



## Why AG News?

AG News is still a sequence-classification problem, so the workflow is almost identical to the sentiment notebook. The difference is that the label space is now multiclass, which makes evaluation and comparison more interesting.

The Hugging Face dataset card reports **120,000 training** examples and **7,600 test** examples for AG News. In this exercise notebook we use a subset.

### Load AG News

Run the dataset load first. Then inspect the available splits.

### Task

Load the AG News dataset and inspect the available splits.

In [11]:
news = load_dataset('ag_news')

### Task

Show the size of each split.

In [10]:
news

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

### Task

Show the label IDs and label names.

In [20]:

id2label = {i: name for i, name in enumerate(news['train'].features['label'].names)}
label2id = {name: i for i, name in id2label.items()}

pd.DataFrame({
    'label_id': list(id2label.keys()),
    'label_name': list(id2label.values()),
})

,label_id,label_name
0,0,World
1,1,Sports
2,2,Business
3,3,Sci/Tech


### Task

Create a smaller train/validation split from the original training data.

Instructions:

- shuffle the original training split,
- select 5000 examples,
- split them 80/20 into `train_small` and `valid_small`.

In [ ]:
news['train'] = news['train'].shuffle(seed=SEED)
news['small'] = news['train'].select(range(5000))


### Task

Create a smaller test set called `test_small` using 1500 examples from the original test split.

## Tokenization

The overall pattern is unchanged from the previous notebook. We still load a tokenizer, inspect tokenization, and prepare model-ready inputs.

### Keep the checkpoint fixed

Use `bert-base-uncased` so the main change is the task and label space, not the model family.

In [ ]:
checkpoint = 'bert-base-uncased'
checkpoint

### Load tokenizer


### Task

Complete the starter code using the checkpoint above.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    ...,   # TODO: checkpoint name
)

tokenizer

### Choose a `MAX_LENGTH`

Pick a value and justify it briefly.

In [ ]:
MAX_LENGTH = ...

### Task

Create a tokenizer for the `text` field with truncation enabled and the chosen `MAX_LENGTH`.

In [ ]:
def tokenize_batch(batch):
    tokenized_batch = ...
    return tokenized_batch

### Task

Map your function across the train, validation, and test subsets.

In [ ]:
tokenized_train = ...
tokenized_valid = ...
tokenized_test = ...

### Task

Inspect one tokenized example

Make sure you can see `input_ids` and `attention_mask`.

### Task

Rename `label` to `labels`

The `Trainer` expects the supervised target column to be called `labels`.

In [ ]:
tokenized_train = ...
tokenized_valid = ...
tokenized_test = ...

### Task

Choose columns and set dataset format to Pytorch

Keep only the columns needed for PyTorch training.

In [ ]:
columns_for_trainer = ['input_ids', 'attention_mask', 'labels']
for dataset in (..., ..., ...):
    ...

# Inspect one formatted item.
...

### Task

Set up the data collator

Use dynamic padding rather than global fixed padding.

In [ ]:
data_collator = ...
data_collator

### Task

Define `id2label` and `label2id`

Build the label mappings from the AG News label names.

In [ ]:
label_names = ...
id2label = ...
label2id = ...
num_labels = ...

pd.DataFrame({
    'label_id': list(id2label.keys()),
    'label_name': list(id2label.values()),
})

### Task  

Define metrics

Because this is multiclass classification, macro F1 matters even more naturally than it did in the binary case.

### Task

Load `AutoModelForSequenceClassification`

Use the same checkpoint, but set the number of labels and label mappings for AG News.

In [ ]:
from transformers import AutoModelForSequenceClassification

baseline_model = AutoModelForSequenceClassification.from_pretrained(
    ...,            # TODO: checkpoint
    num_labels=...,
    id2label=...,
    label2id=...,
    trust_remote_code=False,
    use_safetensors=True,
)

### Task

Freeze the backbone

Loop through the encoder parameters and set `requires_grad = False`.

In [ ]:
for param in ...:
    ...

#### Hint

For BERT, the simplest route is to work with `baseline_model.base_model.parameters()`.

### Task

Define training arguments for the frozen-head baseline.

In [ ]:
from transformers import TrainingArguments

baseline_training_args = TrainingArguments(
    output_dir='assets/baseline_news',
    learning_rate=...,
    per_device_train_batch_size=...,
    per_device_eval_batch_size=...,
    num_train_epochs=...,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    seed=SEED,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
)

### Task

Build the baseline trainer

Use the tokenized train and validation sets here. Do not touch the test set yet.

In [ ]:
from transformers import Trainer

baseline_trainer = Trainer(
    model=...,
    args=...,
    train_dataset=...,
    eval_dataset=...,
    tokenizer=...,
    data_collator=...,
    compute_metrics=...,
)

### Task

Run the baseline

Train the frozen-head model.

In [ ]:
baseline_train_result = ...
baseline_train_result

### Task

Evaluate the baseline

Use the **validation** split here. We are not using the test set for model comparison yet.

### Task

Classification report (baseline)

Print the report for the validation set.

In [ ]:
print(classification_report(
    ...,
    ...,
    target_names=...,
))

### Task

Plot the confusion matrix (baseline)

This is where the multiclass setup becomes especially useful.

In [ ]:
baseline_cm = ...

plt.figure(figsize=(5.2, 4.6))
sns.heatmap(
    baseline_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=...,
    yticklabels=...,
)
plt.title('Frozen-head baseline confusion matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

## Full fine-tuning

Now reload a fresh BERT model and repeat the same pattern, but with all weights trainable and a smaller learning rate.

### Task

Load a fresh model

Do not reuse the frozen baseline object.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    ...,            # TODO: checkpoint
    num_labels=...,
    id2label=...,
    label2id=...,
    trust_remote_code=False,
    use_safetensors=True,
)

### Task

Fill in the fine-tuning arguments

Use a smaller learning rate than the frozen-head baseline.

In [ ]:
training_args = TrainingArguments(
    output_dir='week6_agnews_full_finetune_runs',
    learning_rate=...,
    per_device_train_batch_size=...,
    per_device_eval_batch_size=...,
    num_train_epochs=...,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    seed=SEED,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
)

### Task

Build and run the fine-tuning trainer

Repeat the same pattern as the baseline section.

In [ ]:
trainer = Trainer(
    model=...,
    args=...,
    train_dataset=...,
    eval_dataset=...,
    tokenizer=...,
    data_collator=...,
    compute_metrics=...,
)

train_result = ...
train_result

### Task

Evaluate on the validation split

Again, stay on validation for the comparison stage.

In [ ]:
valid_pred_out = ...
valid_pred_ids = ...

full_valid_acc = ...
full_valid_f1 = ...

full_validation_summary = pd.DataFrame([
    {
        'split': 'validation',
        'accuracy': full_valid_acc,
        'macro_f1': full_valid_f1,
    }
])
full_validation_summary

## Compare both models

This is the key comparison stage. Use **validation** metrics here, not test metrics.

### Task

Create a comparison table

Include validation accuracy, validation macro F1, trainable fraction, and one runtime note.

In [ ]:
baseline_param_summary = parameter_summary(baseline_model)
full_param_summary = parameter_summary(model)

comparison_storyboard = pd.DataFrame([
    {
        'model': 'bert_frozen_head',
        'validation_accuracy': ...,
        'validation_macro_f1': ...,
        'trainable_fraction': ...,
        'runtime_note': '',
    },
    {
        'model': 'bert_full_finetune',
        'validation_accuracy': ...,
        'validation_macro_f1': ...,
        'trainable_fraction': ...,
        'runtime_note': '',
    },
])
comparison_storyboard

### Task

Decide which model to take to test

Write one sentence explaining which model you would choose based on validation evidence.

## Final test evaluation

Now evaluate **one chosen model** on the test set once.

### Task

Test the model you think is best on the test set. The chosen model name and trainer based on your validation decision.

In [ ]:
chosen_test_pred_out = ...
chosen_test_pred_ids = ...

chosen_test_acc = ...
chosen_test_f1 = ...

final_test_summary = pd.DataFrame([
    {
        'model': chosen_model_name,
        'test_accuracy': chosen_test_acc,
        'test_macro_f1': chosen_test_f1,
    }
])
final_test_summary

### Task

Classification report and confusion matrix (chosen model)

Use the test labels only here.

In [ ]:
print(classification_report(
    ...,
    ...,
    target_names=...,
))

chosen_test_cm = ...
plt.figure(figsize=(5.2, 4.6))
sns.heatmap(
    chosen_test_cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=...,
    yticklabels=...,
)
plt.title('Chosen model test confusion matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

## Error analysis and reflection

This is where you turn scores into judgement.

### Task

Inspect a few wrong predictions from the chosen model

Create a dataframe with text, true label, predicted label, and confidence.

In [ ]:
probs = ...
conf = ...

chosen_results_df = pd.DataFrame({
    'text': ...,
    'true_label': ...,
    'pred_label': ...,
    'confidence': ...,
})
chosen_results_df['true_label_name'] = ...
chosen_results_df['pred_label_name'] = ...

chosen_errors_df = ...
chosen_errors_df.head(10)

### Task

Final reflection

Write a short final reflection.

Answer these questions:

1. Which model won on validation?
2. Was full fine-tuning worth it?
3. Which classes were easiest?
4. Which classes got confused?
5. What limitation still remains?